In [0]:
%sql
CREATE OR REPLACE TABLE olist.gold.monthly_delivery_performance
USING DELTA
AS
SELECT
  purchase_month,
  customer_state,
  COUNT(*) AS order_count,
  COUNT(CASE WHEN order_status = 'delivered' THEN 1 END) AS delivered_orders,
  COUNT(days_vs_estimate) AS eligible_deliveries,
  COUNT(CASE WHEN days_vs_estimate > 0 THEN 1 END) AS late_deliveries,
  SUM(CASE WHEN days_vs_estimate > 0 THEN days_vs_estimate ELSE 0 END) AS total_days_late
FROM olist.silver.orders_enriched
GROUP BY purchase_month, customer_state

num_affected_rows,num_inserted_rows


In [0]:
%sql
WITH silver_totals AS (
  SELECT
    purchase_month,
    COUNT(*) AS order_count,
    COUNT(days_vs_estimate) AS eligible_deliveries,
    COUNT(CASE WHEN days_vs_estimate > 0 THEN 1 END) AS late_deliveries
  FROM olist.silver.orders_enriched
  GROUP BY purchase_month
),
gold_totals AS (
  SELECT
    purchase_month,
    SUM(order_count) AS order_count,
    SUM(eligible_deliveries) AS eligible_deliveries,
    SUM(late_deliveries) AS late_deliveries
  FROM olist.gold.monthly_delivery_performance
  GROUP BY purchase_month
)
SELECT assert_true(
  COUNT(*) = 0,
  'Gold check failed: monthly totals differ from Silver.'
) AS gold_quality_check
FROM silver_totals AS s
FULL OUTER JOIN gold_totals AS g
  ON s.purchase_month = g.purchase_month
WHERE s.purchase_month IS NULL
   OR g.purchase_month IS NULL
   OR s.order_count <> g.order_count
   OR s.eligible_deliveries <> g.eligible_deliveries
   OR s.late_deliveries <> g.late_deliveries

gold_quality_check
null
